## 1️⃣ Imports & Configuration

In [20]:
# ✅ Importer les bibliothèques nécessaires
import os
import ee
import geopandas as gpd
import pandas as pd
import xarray as xr
import rioxarray
from shapely.geometry import Point
from rasterstats import zonal_stats

In [ ]:
# ✅ Définir le dossier de sortie
output_dir = r"C:\Users\tchio\OneDrive\Bureau\data science\projet biscaye\extration of climate data"
os.makedirs(output_dir, exist_ok=True)  # S'assurer que le dossier existe

print(f"✅ Tous les fichiers seront enregistrés dans : {output_dir}")

## 3️⃣  fichiers NetCDF 

In [22]:
# ✅ Définition des fichiers NetCDF
nc_cloud = r"C:\Users\tchio\OneDrive\Bureau\data science\projet memoire m2\donnée sur les températures et précipitations Esaie\cru_ts4.08.1901.2023.cld.dat.nc\cru_ts4.08.1901.2023.cld.dat.nc"
nc_dtr   = r"C:\Users\tchio\OneDrive\Bureau\data science\projet memoire m2\donnée sur les températures et précipitations Esaie\cru_ts4.08.1901.2023.dtr.dat.nc\cru_ts4.08.1901.2023.dtr.dat.nc"
nc_frost = r"C:\Users\tchio\OneDrive\Bureau\data science\projet memoire m2\donnée sur les températures et précipitations Esaie\cru_ts4.08.1901.2023.frs.dat.nc\cru_ts4.08.1901.2023.frs.dat.nc"
nc_pet   = r"C:\Users\tchio\OneDrive\Bureau\data science\projet memoire m2\donnée sur les températures et précipitations Esaie\cru_ts4.08.1901.2023.pet.dat.nc\cru_ts4.08.1901.2023.pet.dat.nc"

In [23]:
# ✅ Charger les fichiers NetCDF et extraire les variables
ds_cloud  = xr.open_dataset(nc_cloud)
ds_dtr    = xr.open_dataset(nc_dtr)
ds_frost  = xr.open_dataset(nc_frost)
ds_pet    = xr.open_dataset(nc_pet)

In [24]:
# ✅ Sélection des variables
da_cloud  = ds_cloud["cld"]    # Couverture nuageuse
da_dtr    = ds_dtr["dtr"]      # Amplitude thermique journalière (Diurnal Temperature Range)
da_frost  = ds_frost["frs"]    # Jours de gel (Frost days)
da_pet    = ds_pet["pet"]      # Évapotranspiration potentielle (Potential Evapotranspiration)

## 4️⃣ Chargement de la grille et des mines

In [ ]:
# ✅ Définir le chemin de la grille subdivisée (mise à jour)
grid_file = r"C:\Users\tchio\OneDrive\Bureau\data science\projet biscaye\extration of climate data\reduce_10km_grid\reduced_10km_grid.shp"

# ✅ Charger la grille subdivisée avec Geopandas
grid_gdf = gpd.read_file(grid_file)

# ✅ Vérifier le chargement
print(f"✅ Grille subdivisée chargée avec succès ! Nombre de cellules : {len(grid_gdf)}")
grid_gdf.head()  # Afficher les premières lignes


In [26]:
# ✅ Charger les données des mines
file_path = r"C:\Users\tchio\OneDrive\Bureau\data science\projet biscaye\extraire coordonnées géo_mines\données_geocodés\données_geocodés\Données géocodées\données_geocodés\merged_data.xlsx"
mine_locations = pd.read_excel(file_path, sheet_name="Sheet1")

In [27]:
# ✅ Supprimer les doublons
mine_locations = mine_locations.drop_duplicates(subset=["Mine", "Latitude", "Longitude"]).dropna().reset_index(drop=True)


In [ ]:
# ✅ Convertir en GeoDataFrame
mine_gdf = gpd.GeoDataFrame(
    mine_locations,
    geometry=[Point(lon, lat) for lat, lon in zip(mine_locations["Latitude"], mine_locations["Longitude"])],
    crs="EPSG:4326"
)

print(f"✅ Mines chargées : {len(mine_gdf)} sites miniers trouvés.")

##  5️⃣ Association des mines aux cellules

In [29]:
# ✅ Associer chaque mine à la cellule où elle se trouve
mines_with_cells = gpd.sjoin(mine_gdf, grid_gdf, how="left", predicate="within")

In [30]:
# ✅ Vérifier si certaines mines ne sont pas associées à une cellule
missing_mines = mines_with_cells[mines_with_cells["index_right"].isna()]
if not missing_mines.empty:
    print(f"⚠ {len(missing_mines)} mines ne sont pas dans la grille !")

In [ ]:
# ✅ Supprimer les mines non associées
mines_with_cells = mines_with_cells.dropna(subset=["index_right"])

print(f"✅ Mines associées aux cellules : {len(mines_with_cells)} mines traitées.")

## 6️⃣ Extraction des données climatiques pour chaque mine

In [ ]:
# ✅ Définir la période d'extraction
start_year = 1901
end_year = 2023

# ✅ Initialiser une liste pour stocker les données mensuelles
climate_records = []

print(f"✅ Début de l'extraction des données climatiques de {start_year} à {end_year}...")

# ✅ Récupérer la dimension temporelle (la même pour toutes les variables)
time_dim = da_cloud["time"]

# ✅ Parcourir chaque mois dans la plage temporelle
for i, time_label in enumerate(time_dim.values):
    date_label = str(time_label)[:7]  # Format YYYY-MM
    print(f"🔄 Extraction des données pour {date_label}...")

    # ✅ Vérifier que la date est bien dans la plage attendue
    if date_label < "1901-01" or date_label > "2023-12":
        print(f"❌ Erreur : La date {date_label} est hors de la plage attendue !")
        continue  # Passer à la prochaine itération si la date est incorrecte

    # ✅ Extraire les valeurs climatiques pour chaque mine
    for _, row in mines_with_cells.iterrows():
        lat, lon, mine_name = row["Latitude"], row["Longitude"], row["Mine"]

        # ✅ Initialiser un dictionnaire pour stocker les valeurs extraites
        climate_data = {
            "Mine": mine_name,
            "Latitude": lat,
            "Longitude": lon,
            "Date": date_label
        }

        # ✅ Extraire les valeurs climatiques directement depuis NetCDF
        for var, da in zip(["Cloud_Cover", "Diurnal_Temp_Range", "Frost_Days", "Potential_Evapotranspiration"],
                           [da_cloud, da_dtr, da_frost, da_pet]):
            value = da.sel(lon=lon, lat=lat, time=time_label, method="nearest").values.item()
            climate_data[var] = value if value is not None else 0  # Remplacer NaN par 0

        # ✅ Ajouter les données extraites à la liste
        climate_records.append(climate_data)

print("✅ Extraction mensuelle des nouvelles variables climatiques terminée.")

# ✅ Convertir en DataFrame
climate_df = pd.DataFrame(climate_records)

# ✅ Trier les données pour un bon ordre temporel
climate_df = climate_df.sort_values(by=["Mine", "Date"]).reset_index(drop=True)

# ✅ Définir le nom du fichier CSV en mentionnant les variables incluses
climate_output_file = os.path.join(output_dir, "mine_climate_cld_dtr_frs_pet_1901_2023.csv")

# ✅ Sauvegarder les données en CSV
climate_df.to_csv(climate_output_file, index=False)
print(f"✅ Extraction terminée ! Données enregistrées sous {climate_output_file}")
